# Standalone Database Table Wipe & Load Tool

This notebook connects to a PostgreSQL database and allows you to:
1. Select either `apg_catalog` or `apg_content` table
2. Wipe all existing data from the selected table (with confirmations)
3. Load new data from a deployment CSV file created by Stage 5

**This version is self-contained and does not require the `iris` project structure.** You only need `pandas`, `psycopg2`, and `ipywidgets` installed.

**⚠️ WARNING**: This tool will DELETE ALL DATA in the selected table before loading new data. Use with caution!

## 1. Configuration

**Important:** Update the `DB_PARAMS` dictionary below with your actual database connection details.

In [ ]:
# --- Database Connection Parameters --- 
# !!! MODIFY THESE VALUES TO MATCH YOUR DATABASE SETUP !!!
DB_PARAMS = {
    "host": "localhost",      # e.g., 'localhost' or an IP address
    "port": "5432",           # Default PostgreSQL port
    "dbname": "maven-finance",  # Your database name
    "user": "iris_dev",       # Your database username
    "password": ""             # Your database password (leave empty if none)
}

# --- Supported Tables ---
SUPPORTED_TABLES = ["apg_catalog", "apg_content"]

# --- End Configuration ---

## 2. Setup and Imports

In [ ]:
import pandas as pd
import psycopg2
import psycopg2.extras
from psycopg2 import sql
import ipywidgets as widgets
from IPython.display import display, clear_output
import os
import logging
import json
from datetime import datetime
import io
from typing import Optional, Dict, Any, Tuple

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger()

# Register UUID adapter
try:
    psycopg2.extras.register_uuid()
except Exception as e:
    logger.warning(f"Could not register UUID adapter: {e}")

## 3. Database Connection Functions

In [ ]:
# Create widgets
table_selector = widgets.Dropdown(
    options=SUPPORTED_TABLES,
    description='Select Table:',
    disabled=False,
    style={'description_width': 'initial'}
)

file_upload = widgets.FileUpload(
    accept='.csv',
    multiple=False,
    description='Upload CSV:',
    style={'description_width': 'initial'}
)

preview_button = widgets.Button(
    description='Preview Data',
    disabled=True,
    button_style='info',
    tooltip='Preview the first 10 rows of the CSV',
    icon='eye'
)

wipe_button = widgets.Button(
    description='Wipe Table',
    disabled=True,
    button_style='danger',
    tooltip='Delete all data from the selected table',
    icon='trash'
)

load_button = widgets.Button(
    description='Load Data',
    disabled=True,
    button_style='success',
    tooltip='Load CSV data into the table',
    icon='upload'
)

output_area = widgets.Output()

# Global variables
current_df = None
conn = None


def on_file_upload_change(change):
    """Handle file upload."""
    global current_df
    
    output_area.clear_output()
    
    if len(file_upload.value) > 0:
        with output_area:
            try:
                # Read the uploaded file
                uploaded_file = list(file_upload.value.values())[0]
                content = uploaded_file['content']
                
                # Parse CSV
                current_df = pd.read_csv(io.BytesIO(content))
                print(f"✅ Loaded CSV with {len(current_df)} rows and {len(current_df.columns)} columns")
                
                # Validate for selected table
                is_valid, issues = validate_csv_for_table(current_df, table_selector.value)
                
                if issues:
                    print("\n⚠️ Validation issues:")
                    for issue in issues:
                        print(f"  - {issue}")
                
                if is_valid or (len(issues) == 1 and "auto-generated fields" in issues[0]):
                    preview_button.disabled = False
                    wipe_button.disabled = False
                    print("\n✅ CSV is compatible with the selected table")
                else:
                    preview_button.disabled = True
                    wipe_button.disabled = True
                    print("\n❌ CSV is not compatible with the selected table")
                    
            except Exception as e:
                print(f"❌ Error reading CSV: {e}")
                current_df = None
                preview_button.disabled = True
                wipe_button.disabled = True
                load_button.disabled = True


def on_preview_click(b):
    """Show preview of the data."""
    if current_df is not None:
        with output_area:
            print(f"\nPreview of data for table: {table_selector.value}")
            print(f"Total rows: {len(current_df)}")
            print(f"\nFirst 10 rows:")
            display(current_df.head(10))
            
            # Show column info
            print("\nColumn information:")
            for col in current_df.columns:
                non_null = current_df[col].notna().sum()
                print(f"  - {col}: {current_df[col].dtype} ({non_null}/{len(current_df)} non-null)")


def on_wipe_click(b):
    """Handle table wipe with confirmation."""
    global conn
    
    with output_area:
        print(f"\n⚠️ WARNING: You are about to DELETE ALL DATA from table '{table_selector.value}'")
        print("\nThis action cannot be undone!")
        
        # Connect to database
        if conn is None or conn.closed:
            conn = connect_to_db(DB_PARAMS)
            if conn is None:
                print("\n❌ Failed to connect to database")
                return
        
        # Show current record count
        current_count = get_table_count(conn, table_selector.value)
        if current_count >= 0:
            print(f"\nCurrent record count: {current_count}")
        
        # Create confirmation widgets
        confirm_check = widgets.Checkbox(
            value=False,
            description=f'I understand that ALL DATA in {table_selector.value} will be permanently deleted',
            disabled=False,
            style={'description_width': 'initial'}
        )
        
        confirm_button = widgets.Button(
            description='Confirm Delete',
            disabled=True,
            button_style='danger',
            icon='exclamation-triangle'
        )
        
        cancel_button = widgets.Button(
            description='Cancel',
            button_style='primary',
            icon='times'
        )
        
        def on_checkbox_change(change):
            confirm_button.disabled = not change['new']
        
        def on_confirm_click(b):
            with output_area:
                print(f"\n🗑️ Wiping table '{table_selector.value}'...")
                
                if wipe_table(conn, table_selector.value):
                    print(f"\n✅ Successfully wiped table '{table_selector.value}'")
                    load_button.disabled = False
                    wipe_button.disabled = True
                else:
                    print(f"\n❌ Failed to wipe table '{table_selector.value}'")
        
        def on_cancel_click(b):
            with output_area:
                print("\n❌ Table wipe cancelled")
        
        confirm_check.observe(on_checkbox_change, names='value')
        confirm_button.on_click(on_confirm_click)
        cancel_button.on_click(on_cancel_click)
        
        display(widgets.VBox([
            confirm_check,
            widgets.HBox([confirm_button, cancel_button])
        ]))


def on_load_click(b):
    """Handle data loading."""
    global conn, current_df
    
    with output_area:
        print(f"\n📤 Loading data into table '{table_selector.value}'...")
        
        if current_df is None:
            print("❌ No data to load. Please upload a CSV file first.")
            return
        
        # Connect to database if needed
        if conn is None or conn.closed:
            conn = connect_to_db(DB_PARAMS)
            if conn is None:
                print("\n❌ Failed to connect to database")
                return
        
        # Preprocess the data
        print("\nPreprocessing data...")
        processed_df = preprocess_dataframe(current_df, table_selector.value)
        
        # Load the data
        print(f"Loading {len(processed_df)} records...")
        success, records_loaded = load_data_to_table(conn, processed_df, table_selector.value)
        
        if success:
            print(f"\n✅ Successfully loaded {records_loaded} records into '{table_selector.value}'")
            
            # Show final count
            final_count = get_table_count(conn, table_selector.value)
            if final_count >= 0:
                print(f"\nTable now contains {final_count} records")
            
            # Disable load button to prevent duplicate loads
            load_button.disabled = True
            print("\n✅ Data loading complete!")
        else:
            print(f"\n❌ Failed to load data into '{table_selector.value}'")
            print("Check the logs above for error details.")


def on_table_change(change):
    """Handle table selection change."""
    if current_df is not None:
        # Re-validate for new table
        on_file_upload_change(None)


# Wire up event handlers
file_upload.observe(on_file_upload_change, names='value')
table_selector.observe(on_table_change, names='value')
preview_button.on_click(on_preview_click)
wipe_button.on_click(on_wipe_click)
load_button.on_click(on_load_click)

# Display the interface
display(widgets.VBox([
    widgets.HTML("<h3>Database Table Wipe & Load Tool</h3>"),
    widgets.HTML("<p><b>⚠️ Warning:</b> This tool will DELETE ALL DATA in the selected table before loading new data!</p>"),
    table_selector,
    file_upload,
    widgets.HBox([preview_button, wipe_button, load_button]),
    output_area
]))

In [ ]:
# Optional: Manual file processing button (use if file upload doesn't trigger automatically)
process_file_button = widgets.Button(
    description='Process Uploaded File',
    button_style='warning',
    tooltip='Click if the file upload did not process automatically',
    icon='refresh'
)

def on_process_file_click(b):
    """Manually trigger file processing."""
    if len(file_upload.value) > 0:
        on_file_upload_change(None)
    else:
        with output_area:
            print("No file uploaded yet. Please select a CSV file.")

process_file_button.on_click(on_process_file_click)

# Display the manual trigger button
display(widgets.VBox([
    widgets.HTML("<h4>Troubleshooting</h4>"),
    widgets.HTML("<p>If the file doesn't process automatically after upload, click the button below:</p>"),
    process_file_button
]))

## 4. CSV Validation Functions

In [ ]:
def validate_csv_for_table(df: pd.DataFrame, table_name: str) -> Tuple[bool, list]:
    """
    Validate that the CSV data matches the expected schema for the table.
    
    Args:
        df: DataFrame containing the CSV data
        table_name: Name of the target table
        
    Returns:
        Tuple of (is_valid, list_of_issues)
    """
    issues = []
    
    # Define required columns for each table (excluding auto-generated fields)
    required_columns = {
        "apg_catalog": [
            "document_source", "document_type", "document_name"
        ],
        "apg_content": [
            "document_source", "document_type", "document_name",
            "section_id", "section_content"
        ]
    }
    
    # Check for required columns
    if table_name in required_columns:
        missing_cols = [col for col in required_columns[table_name] if col not in df.columns]
        if missing_cols:
            issues.append(f"Missing required columns: {', '.join(missing_cols)}")
    
    # Check for empty dataframe
    if df.empty:
        issues.append("CSV file is empty")
    
    # Check for auto-generated fields that should not be in the CSV
    auto_fields = ["id", "created_at"]
    present_auto_fields = [field for field in auto_fields if field in df.columns]
    if present_auto_fields:
        issues.append(f"CSV contains auto-generated fields that will be ignored: {', '.join(present_auto_fields)}")
    
    return (len(issues) == 0, issues)


def preprocess_dataframe(df: pd.DataFrame, table_name: str) -> pd.DataFrame:
    """
    Preprocess the DataFrame to match PostgreSQL requirements.
    
    Args:
        df: DataFrame to preprocess
        table_name: Name of the target table
        
    Returns:
        Preprocessed DataFrame
    """
    df_processed = df.copy()
    
    # Remove any auto-generated fields if present
    auto_fields = ["id", "created_at"]
    for field in auto_fields:
        if field in df_processed.columns:
            df_processed = df_processed.drop(field, axis=1)
            logger.info(f"Removed auto-generated field: {field}")
    
    # Handle numeric fields
    numeric_fields = {
        "apg_catalog": ["file_size"],
        "apg_content": ["section_id", "page_number"]
    }
    
    if table_name in numeric_fields:
        for field in numeric_fields[table_name]:
            if field in df_processed.columns:
                # Convert to numeric, replacing empty strings and 'NULL' with NaN
                df_processed[field] = df_processed[field].replace(['', 'NULL'], pd.NA)
                df_processed[field] = pd.to_numeric(df_processed[field], errors='coerce')
    
    # Handle timestamp fields
    timestamp_fields = ["date_created", "date_last_modified"]
    for field in timestamp_fields:
        if field in df_processed.columns:
            # Convert 'NULL' strings to NaN
            df_processed[field] = df_processed[field].replace('NULL', pd.NA)
            # Parse timestamps
            df_processed[field] = pd.to_datetime(df_processed[field], errors='coerce')
    
    # Handle embedding fields (should be JSON strings or NULL)
    embedding_fields = ["document_usage_embedding", "document_description_embedding"]
    for field in embedding_fields:
        if field in df_processed.columns:
            # Replace 'NULL' strings with None
            df_processed[field] = df_processed[field].replace('NULL', None)
            # Ensure empty strings become None
            df_processed[field] = df_processed[field].replace('', None)
    
    # Clean text fields
    text_fields = ["document_description", "document_usage", "section_summary", "section_content",
                   "section_name", "document_source", "document_type", "document_name",
                   "file_name", "file_type", "file_path", "file_link"]
    for field in text_fields:
        if field in df_processed.columns:
            # Replace 'NULL' strings with None
            df_processed[field] = df_processed[field].replace('NULL', None)
            # Remove null bytes
            df_processed[field] = df_processed[field].astype(str).str.replace('\x00', '', regex=False)
            # Replace 'nan' strings with None
            df_processed[field] = df_processed[field].replace('nan', None)
    
    return df_processed

## 5. Data Loading Functions

In [ ]:
def wipe_table(conn: psycopg2.extensions.connection, table_name: str) -> bool:
    """
    Delete all data from the specified table.
    
    Args:
        conn: Database connection
        table_name: Name of the table to wipe
        
    Returns:
        True if successful, False otherwise
    """
    try:
        with conn.cursor() as cur:
            # Get count before deletion
            count_before = get_table_count(conn, table_name)
            logger.info(f"Table {table_name} has {count_before} records before deletion")
            
            # Delete all records
            delete_query = sql.SQL("DELETE FROM {}").format(sql.Identifier(table_name))
            cur.execute(delete_query)
            
            # Get count after deletion
            count_after = get_table_count(conn, table_name)
            logger.info(f"Table {table_name} has {count_after} records after deletion")
            
            # Commit the transaction
            conn.commit()
            logger.info(f"Successfully wiped table {table_name}")
            return True
            
    except Exception as e:
        logger.error(f"Error wiping table {table_name}: {e}")
        conn.rollback()
        return False


def load_data_to_table(conn: psycopg2.extensions.connection, df: pd.DataFrame, table_name: str) -> Tuple[bool, int]:
    """
    Load data from DataFrame to the specified table using COPY.
    
    Args:
        conn: Database connection
        df: DataFrame containing the data
        table_name: Name of the target table
        
    Returns:
        Tuple of (success, number_of_records_loaded)
    """
    try:
        # Create a CSV buffer
        csv_buffer = io.StringIO()
        df.to_csv(csv_buffer, index=False, header=True, na_rep='\\N')
        csv_buffer.seek(0)
        
        with conn.cursor() as cur:
            # Get column names from DataFrame
            columns = df.columns.tolist()
            
            # Use COPY to load data
            copy_query = sql.SQL("COPY {} ({}) FROM STDIN WITH (FORMAT CSV, HEADER TRUE, NULL '\\N')").format(
                sql.Identifier(table_name),
                sql.SQL(', ').join(map(sql.Identifier, columns))
            )
            
            cur.copy_expert(copy_query, csv_buffer)
            
            # Get count after loading
            count_after = get_table_count(conn, table_name)
            
            # Commit the transaction
            conn.commit()
            logger.info(f"Successfully loaded {len(df)} records to table {table_name}")
            
            return (True, len(df))
            
    except Exception as e:
        logger.error(f"Error loading data to table {table_name}: {e}", exc_info=True)
        conn.rollback()
        return (False, 0)

## 6. Interactive Widget Interface

In [ ]:
# Create widgets
table_selector = widgets.Dropdown(
    options=SUPPORTED_TABLES,
    description='Select Table:',
    disabled=False,
    style={'description_width': 'initial'}
)

file_upload = widgets.FileUpload(
    accept='.csv',
    multiple=False,
    description='Upload CSV:',
    style={'description_width': 'initial'}
)

preview_button = widgets.Button(
    description='Preview Data',
    disabled=True,
    button_style='info',
    tooltip='Preview the first 10 rows of the CSV',
    icon='eye'
)

wipe_button = widgets.Button(
    description='Wipe Table',
    disabled=True,
    button_style='danger',
    tooltip='Delete all data from the selected table',
    icon='trash'
)

load_button = widgets.Button(
    description='Load Data',
    disabled=True,
    button_style='success',
    tooltip='Load CSV data into the table',
    icon='upload'
)

output_area = widgets.Output()

# Global variables
current_df = None
conn = None


def on_file_upload_change(change):
    """Handle file upload."""
    global current_df
    
    output_area.clear_output()
    
    if file_upload.value:
        with output_area:
            try:
                # Read the uploaded file
                uploaded_file = list(file_upload.value.values())[0]
                content = uploaded_file['content']
                
                # Parse CSV
                current_df = pd.read_csv(io.BytesIO(content))
                print(f"✅ Loaded CSV with {len(current_df)} rows and {len(current_df.columns)} columns")
                
                # Validate for selected table
                is_valid, issues = validate_csv_for_table(current_df, table_selector.value)
                
                if issues:
                    print("\n⚠️ Validation issues:")
                    for issue in issues:
                        print(f"  - {issue}")
                
                if is_valid or (len(issues) == 1 and "auto-generated fields" in issues[0]):
                    preview_button.disabled = False
                    wipe_button.disabled = False
                    print("\n✅ CSV is compatible with the selected table")
                else:
                    preview_button.disabled = True
                    wipe_button.disabled = True
                    print("\n❌ CSV is not compatible with the selected table")
                    
            except Exception as e:
                print(f"❌ Error reading CSV: {e}")
                current_df = None
                preview_button.disabled = True
                wipe_button.disabled = True
                load_button.disabled = True


def on_preview_click(b):
    """Show preview of the data."""
    if current_df is not None:
        with output_area:
            clear_output()
            print(f"\nPreview of data for table: {table_selector.value}")
            print(f"Total rows: {len(current_df)}")
            print(f"\nFirst 10 rows:")
            display(current_df.head(10))
            
            # Show column info
            print("\nColumn information:")
            for col in current_df.columns:
                non_null = current_df[col].notna().sum()
                print(f"  - {col}: {current_df[col].dtype} ({non_null}/{len(current_df)} non-null)")


def on_wipe_click(b):
    """Handle table wipe with confirmation."""
    global conn
    
    with output_area:
        clear_output()
        
        # First confirmation
        print(f"⚠️ WARNING: You are about to DELETE ALL DATA from table '{table_selector.value}'")
        print("\nThis action cannot be undone!")
        
        # Connect to database
        if conn is None or conn.closed:
            conn = connect_to_db(DB_PARAMS)
            if conn is None:
                print("\n❌ Failed to connect to database")
                return
        
        # Show current record count
        current_count = get_table_count(conn, table_selector.value)
        if current_count >= 0:
            print(f"\nCurrent record count: {current_count}")
        
        # Create confirmation widgets
        confirm_check = widgets.Checkbox(
            value=False,
            description=f'I understand that ALL DATA in {table_selector.value} will be permanently deleted',
            disabled=False,
            style={'description_width': 'initial'}
        )
        
        confirm_button = widgets.Button(
            description='Confirm Delete',
            disabled=True,
            button_style='danger',
            icon='exclamation-triangle'
        )
        
        cancel_button = widgets.Button(
            description='Cancel',
            button_style='primary',
            icon='times'
        )
        
        def on_checkbox_change(change):
            confirm_button.disabled = not change['new']
        
        def on_confirm_click(b):
            clear_output()
            print(f"🗑️ Wiping table '{table_selector.value}'...")
            
            if wipe_table(conn, table_selector.value):
                print(f"\n✅ Successfully wiped table '{table_selector.value}'")
                load_button.disabled = False
                wipe_button.disabled = True
            else:
                print(f"\n❌ Failed to wipe table '{table_selector.value}'")
        
        def on_cancel_click(b):
            clear_output()
            print("❌ Table wipe cancelled")
        
        confirm_check.observe(on_checkbox_change, names='value')
        confirm_button.on_click(on_confirm_click)
        cancel_button.on_click(on_cancel_click)
        
        display(widgets.VBox([
            confirm_check,
            widgets.HBox([confirm_button, cancel_button])
        ]))


def on_load_click(b):
    """Handle data loading."""
    global conn, current_df
    
    with output_area:
        clear_output()
        
        if current_df is None:
            print("❌ No data to load. Please upload a CSV file first.")
            return
        
        print(f"📤 Loading data into table '{table_selector.value}'...")
        
        # Connect to database if needed
        if conn is None or conn.closed:
            conn = connect_to_db(DB_PARAMS)
            if conn is None:
                print("\n❌ Failed to connect to database")
                return
        
        # Preprocess the data
        print("\nPreprocessing data...")
        processed_df = preprocess_dataframe(current_df, table_selector.value)
        
        # Load the data
        print(f"Loading {len(processed_df)} records...")
        success, records_loaded = load_data_to_table(conn, processed_df, table_selector.value)
        
        if success:
            print(f"\n✅ Successfully loaded {records_loaded} records into '{table_selector.value}'")
            
            # Show final count
            final_count = get_table_count(conn, table_selector.value)
            if final_count >= 0:
                print(f"\nTable now contains {final_count} records")
            
            # Disable load button to prevent duplicate loads
            load_button.disabled = True
            print("\n✅ Data loading complete!")
        else:
            print(f"\n❌ Failed to load data into '{table_selector.value}'")
            print("Check the logs above for error details.")


def on_table_change(change):
    """Handle table selection change."""
    if current_df is not None:
        # Re-validate for new table
        on_file_upload_change(None)


# Wire up event handlers
file_upload.observe(on_file_upload_change, names='value')
table_selector.observe(on_table_change, names='value')
preview_button.on_click(on_preview_click)
wipe_button.on_click(on_wipe_click)
load_button.on_click(on_load_click)

# Display the interface
display(widgets.VBox([
    widgets.HTML("<h3>Database Table Wipe & Load Tool</h3>"),
    widgets.HTML("<p><b>⚠️ Warning:</b> This tool will DELETE ALL DATA in the selected table before loading new data!</p>"),
    table_selector,
    file_upload,
    widgets.HBox([preview_button, wipe_button, load_button]),
    output_area
]))

## 7. Instructions

### Prerequisites
1. Install required packages: `pip install pandas psycopg2-binary ipywidgets`
2. Ensure you have deployment CSV files from Stage 5 (e.g., `catalog_2024-01-15_10-30-00.csv`)

### Usage Steps
1. **Configure:** Update the `DB_PARAMS` in the first code cell with your database details
2. **Run All Cells:** Execute all cells in the notebook
3. **Select Table:** Choose either `apg_catalog` or `apg_content` from the dropdown
4. **Upload CSV:** Use the file upload widget to select your deployment CSV file
5. **Preview Data:** Click "Preview Data" to see the first 10 rows and column information
6. **Wipe Table:** Click "Wipe Table" and confirm to delete all existing data
7. **Load Data:** Click "Load Data" to import the CSV data into the table

### Important Notes
- The tool will automatically remove `id` and `created_at` fields from the CSV (these are auto-generated)
- Timestamps are expected in ISO format with timezone (e.g., `2024-01-15 10:30:00+00`)
- Embedding columns should contain JSON arrays or be empty/NULL
- The wipe operation requires explicit confirmation and cannot be undone
- All operations are transactional - if loading fails, no partial data will be inserted

## 8. Cleanup

In [ ]:
# Close database connection when done
try:
    if 'conn' in locals() and conn and not conn.closed:
        conn.close()
        logger.info("Database connection closed")
        print("Database connection closed.")
    else:
        print("No active database connection to close.")
except Exception as e:
    logger.error(f"Error closing connection: {e}")
    print(f"Error closing connection: {e}")